# Natural Language Processing Lab
## Experiment 1: Working with Text Input and Python Data Structures
**Domain Application:** Terms and Conditions Summarizer (Amazon & Alibaba Agreements)

### Program 1: Safe File Reading with UTF-8 Encoding
Read unstructured text files (`amazon.txt` & `alibaba.txt`) using robust exception handling.

In [1]:
# Program 1: Safe file reading function for T&C documents
def read_tc_document(file_path):
    try:
        with open(file_path, "r", encoding="utf-8") as file:
            content = file.read()
            print(f"[SUCCESS] Successfully loaded '{file_path}' ({len(content)} characters)")
            return content
    except FileNotFoundError:
        print(f"[ERROR] File not found: {file_path}")
        return None
    except UnicodeDecodeError:
        print(f"[ERROR] Unable to decode file with UTF-8: {file_path}")
        return None

# Read Amazon & Alibaba agreement files
amazon_text = read_tc_document("data/amazon.txt")
alibaba_text = read_tc_document("data/alibaba.txt")

[SUCCESS] Successfully loaded 'data/amazon.txt' (5109 characters)
[SUCCESS] Successfully loaded 'data/alibaba.txt' (4246 characters)


### Program 2: Line-by-Line Ingestion, Tuples, and Unique Vocabulary Sets
Parse legal documents into clauses (`list`), create `(clause_index, word_count)` metadata pairs (`tuple`), and compute unique vocabulary sets (`set`).

In [1]:
# Program 2: Line-by-line reading into list of clauses
def extract_clauses(file_path):
    clauses = []
    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            cleaned = line.strip()
            if cleaned and not cleaned.isupper():  # Filter blank lines and standalone headers
                clauses.append(cleaned)
    return clauses

amazon_clauses = extract_clauses("data/amazon.txt")
alibaba_clauses = extract_clauses("data/alibaba.txt")

print(f"Amazon Clauses Extracted: {len(amazon_clauses)}")
print(f"Alibaba Clauses Extracted: {len(alibaba_clauses)}")

# 1. Store as Tuples: (Clause_Index, Word_Count, First_50_Chars)
amazon_clause_tuples = [(idx + 1, len(c.split()), c[:50] + "...") for idx, c in enumerate(amazon_clauses)]
print("\n--- Sample Amazon Clause Tuples (First 3) ---")
for t in amazon_clause_tuples[:3]:
    print(t)

# 2. Store as Sets: Unique Vocabulary Extraction
amazon_words = set(w.lower().strip(".,()[]\"") for c in amazon_clauses for w in c.split())
alibaba_words = set(w.lower().strip(".,()[]\"") for c in alibaba_clauses for w in c.split())

print(f"\nAmazon Unique Vocabulary Size: {len(amazon_words)}")
print(f"Alibaba Unique Vocabulary Size: {len(alibaba_words)}")
common_legal_terms = amazon_words.intersection(alibaba_words)
print(f"Shared Legal Terms Count: {len(common_legal_terms)}")
print("Sample Shared Terms:", list(common_legal_terms)[:10])

Amazon Clauses Extracted: 29
Alibaba Clauses Extracted: 21

--- Sample Amazon Clause Tuples (First 3) ---
(1, 4, 'Amazon.com Conditions of Use...')
(2, 26, 'Welcome to Amazon.com. Amazon.com Services LLC and...')
(3, 29, 'By using Amazon Services, you agree, on behalf of ...')

Amazon Unique Vocabulary Size: 297
Alibaba Unique Vocabulary Size: 211
Shared Legal Terms Count: 90
Sample Shared Terms: ['with', 'and', 'only', 'has', 'when', 'account', 'cancel', 'agree', 'graphics', 'software']


### Program 3: Structured CSV Ingestion & Dataset Inspection with Pandas
Load `tc_clauses.csv` and inspect dataset dimensions, data types, null values, and category frequencies.

In [1]:
# Program 3: Tabular Data Processing with Pandas
import pandas as pd

# Load structured T&C dataset
df_clauses = pd.read_csv("data/tc_clauses.csv")

print("--- Dataset Shape (Rows, Columns) ---")
print(df_clauses.shape)

print("\n--- First 5 Rows ---")
display(df_clauses.head())

print("\n--- Missing Values Check ---")
print(df_clauses.isnull().sum())

print("\n--- Risk Level Distribution ---")
print(df_clauses["risk_level"].value_counts())

print("\n--- Category Distribution across Companies ---")
print(pd.crosstab(df_clauses["company"], df_clauses["risk_level"]))

# Feature engineering: Add word count & character count columns
df_clauses["word_count"] = df_clauses["clause_text"].apply(lambda x: len(str(x).split()))
df_clauses["char_count"] = df_clauses["clause_text"].apply(lambda x: len(str(x)))

print("\n--- DataFrame with NLP Feature Columns ---")
display(df_clauses[["clause_id", "company", "category", "risk_level", "word_count", "char_count"]].head())

--- Dataset Shape (Rows, Columns) ---
(17, 5)

--- First 5 Rows ---


,clause_id,company,category,clause_text,risk_level
0,AMZ_001,Amazon,Privacy,"Please review our Privacy Notice, which also g...",Low
1,AMZ_002,Amazon,Electronic Communications,You consent to receive communications from us ...,Low
2,AMZ_003,Amazon,Intellectual Property,All content included in or made available thro...,Medium
3,AMZ_004,Amazon,Account Security,You are responsible for maintaining the confid...,Medium
4,AMZ_005,Amazon,Termination,"Amazon reserves the right to refuse service, t...",High



--- Missing Values Check ---
clause_id      0
company        0
category       0
clause_text    0
risk_level     0
dtype: int64

--- Risk Level Distribution ---
risk_level
High      9
Medium    5
Low       3
Name: count, dtype: int64

--- Category Distribution across Companies ---
risk_level  High  Low  Medium
company                      
Alibaba        4    1       3
Amazon         5    2       2

--- DataFrame with NLP Feature Columns ---


,clause_id,company,category,risk_level,word_count,char_count
0,AMZ_001,Amazon,Privacy,Low,17,110
1,AMZ_002,Amazon,Electronic Communications,Low,22,144
2,AMZ_003,Amazon,Intellectual Property,Medium,25,152
3,AMZ_004,Amazon,Account Security,Medium,19,128
4,AMZ_005,Amazon,Termination,High,27,175


### Program 4: Hierarchical JSON Ingestion & Dictionary Mapping
Parse nested `tc_policies.json` and build lookup dictionaries mapping company sections to their policy clauses.

In [2]:
# Program 4: Parsing Nested JSON and Dictionary Operations
import json

with open("data/tc_policies.json", "r", encoding="utf-8") as file:
    policies_data = json.load(file)

# Build lookup dictionary: {Company -> {Section_Name -> [Clauses]}}
policy_lookup = {}

for policy in policies_data["policies"]:
    company = policy["company"]
    policy_lookup[company] = {}
    for sec in policy["sections"]:
        sec_name = sec["section_name"]
        policy_lookup[company][sec_name] = sec["clauses"]

# Inspect dictionary structure
print("Companies Indexed:", list(policy_lookup.keys()))
print("\nAmazon Sections:", list(policy_lookup["Amazon"].keys()))
print("Alibaba Sections:", list(policy_lookup["Alibaba"].keys()))

print("\n--- Specific Lookup: Alibaba Limitation of Liability Clauses ---")
for idx, clause in enumerate(policy_lookup["Alibaba"]["Limitation of Liability"], start=1):
    print(f"{idx}. {clause}")

Companies Indexed: ['Amazon', 'Alibaba']

Amazon Sections: ['Privacy', 'Account Responsibilities', 'Limitation of Liability']
Alibaba Sections: ['Acceptance', 'Termination', 'Limitation of Liability']

--- Specific Lookup: Alibaba Limitation of Liability Clauses ---
1. Services are provided with all faults and without warranties of durability or merchantability.
2. Under no circumstances shall Alibaba.com be held liable for loss of profits or business interruption.


### Extension Activity: Statistical Comparison of Amazon vs Alibaba Clauses
Aggregate and compare average word count and clause length metrics by company and risk level.

In [3]:
# Extension Activity: Comprehensive Summary DataFrame
summary_stats = df_clauses.groupby(["company", "risk_level"]).agg(
    total_clauses=("clause_id", "count"),
    avg_words=("word_count", "mean"),
    avg_chars=("char_count", "mean"),
    max_words=("word_count", "max"),
    min_words=("word_count", "min")
).reset_index()

summary_stats["avg_words"] = summary_stats["avg_words"].round(2)
summary_stats["avg_chars"] = summary_stats["avg_chars"].round(2)

print("--- Summary Statistics by Company and Risk Level ---")
display(summary_stats)

--- Summary Statistics by Company and Risk Level ---


,company,risk_level,total_clauses,avg_words,avg_chars,max_words,min_words
0,Alibaba,High,4,19.75,126.00,21,19
1,Alibaba,Low,1,18.00,92.00,18,18
2,Alibaba,Medium,3,19.00,126.33,21,17
3,Amazon,High,5,23.40,148.00,27,18
4,Amazon,Low,2,19.50,127.00,22,17
5,Amazon,Medium,2,22.00,140.00,25,19
